# Task 2.7 — Held-out Evaluation and ByteTrack Exports

This notebook writes held-out metrics and four tracking JSONL files into Drive. It uses the Task 2.6 YOLO11s checkpoint and dataset, plus `yolo11n.pt` as the COCO baseline.

Before running: put `best.pt` at `MyDrive/VEYAKSH_TRAFFIC_DATASET/models/best.pt`; put `test.mp4` and one campus video at `MyDrive/VEYAKSH_TRAFFIC_DATASET/data/test.mp4` and `campus_clip.mp4`. Adjust the paths in the next cell if your files have different names/locations. Colab GPU is optional for tracking, but recommended for speed.


In [ ]:
%pip -q install -U ultralytics pyyaml opencv-python-headless
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import Counter
import csv, json, cv2, torch
import numpy as np
from ultralytics import YOLO

DRIVE = Path('/content/drive/MyDrive')
VEYAKSH = DRIVE/'VEYAKSH_TRAFFIC_DATASET'
BEST = VEYAKSH/'models/best.pt'
DATA_YAML = DRIVE/'TrafficDetection/results/task2_6_final_yolo11s_v1/traffic_dataset/data.yaml'
TEST_VIDEO = VEYAKSH/'data/test.mp4'
CAMPUS_VIDEO = VEYAKSH/'data/campus_clip.mp4'
TRAIN_RESULTS = DRIVE/'TrafficDetection/results/task2_6_final_yolo11s_v1'
TRAIN_CSV = TRAIN_RESULTS/'yolo11s_task2_6_v1/results.csv'
TASK26_MANIFEST = TRAIN_RESULTS/'task2_6_manifest_v1.csv'
TASK26_METRICS_JSON = TRAIN_RESULTS/'task2_6_heldout_clip_metrics_v1.json'
METRICS_DIR = VEYAKSH/'models/task2_7_eval'
TRACKS_DIR = VEYAKSH/'tracks'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU')
for p in (BEST, DATA_YAML, TEST_VIDEO, CAMPUS_VIDEO):
    assert p.is_file(), f'Missing required input: {p}'
if TRAIN_CSV.is_file():
    import shutil
    shutil.copy2(TRAIN_CSV, VEYAKSH/'models/results.csv')
    print('Copied Task 2.6 training results.csv to', VEYAKSH/'models/results.csv')
METRICS_DIR.mkdir(parents=True, exist_ok=True)
TRACKS_DIR.mkdir(parents=True, exist_ok=True)


## Held-out clip evaluation

The validation `data.yaml` must still point to the same whole clip held out in Task 2.6. This reruns validation and exports aggregate metrics, per-class AP (blank for classes absent from validation ground truth), confusion-matrix CSV, and Ultralytics confusion-matrix plots.


In [ ]:
eval_model = YOLO(str(BEST))
val = eval_model.val(
    data=str(DATA_YAML), split='val', imgsz=640, batch=8,
    device=DEVICE, plots=True, project=str(METRICS_DIR),
    name='heldout_clip_eval', exist_ok=True, verbose=True,
)
box = val.box
names = eval_model.names
if isinstance(names, list): names = dict(enumerate(names))
rows = [
    {'metric':'precision','class_id':'all','class_name':'all','value':float(box.mp)},
    {'metric':'recall','class_id':'all','class_name':'all','value':float(box.mr)},
    {'metric':'mAP50','class_id':'all','class_name':'all','value':float(box.map50)},
    {'metric':'mAP50-95','class_id':'all','class_name':'all','value':float(box.map)},
]
ids = [int(x) for x in getattr(box, 'ap_class_index', [])]
ap50 = list(getattr(box, 'ap50', [])); ap = list(getattr(box, 'ap', []))
per_class = {cid:(float(ap50[i]),float(ap[i])) for i,cid in enumerate(ids)}
for cid in range(7):
    cname = str(names.get(cid, f'class_{cid}'))
    vals = per_class.get(cid)
    rows += [
        {'metric':'AP50','class_id':cid,'class_name':cname,'value':vals[0] if vals else ''},
        {'metric':'AP50-95','class_id':cid,'class_name':cname,'value':vals[1] if vals else ''},
    ]
with (METRICS_DIR/'task2_7_metrics.csv').open('w',newline='',encoding='utf-8-sig') as f:
    w=csv.DictWriter(f,fieldnames=['metric','class_id','class_name','value']); w.writeheader(); w.writerows(rows)
cm = getattr(getattr(val,'confusion_matrix',None),'matrix',None)
if cm is not None: np.savetxt(METRICS_DIR/'task2_7_confusion_matrix.csv',np.asarray(cm),delimiter=',',fmt='%.6f')
summary = {'precision':float(box.mp),'recall':float(box.mr),'mAP50':float(box.map50),'mAP50-95':float(box.map),'validation_split':'val','weights':str(BEST),'dataset_yaml':str(DATA_YAML),'ultralytics_run_dir':str(val.save_dir)}
(METRICS_DIR/'task2_7_eval_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))
print('Metrics:', METRICS_DIR/'task2_7_metrics.csv')
print('Confusion matrix plots:', val.save_dir)


In [ ]:
# Print the exact held-out clip and split counts for DECISIONS.md.
from collections import Counter
if TASK26_METRICS_JSON.is_file():
    task26_metrics=json.loads(TASK26_METRICS_JSON.read_text(encoding='utf-8'))
    print('Exact held-out clip ID:', task26_metrics.get('validation_clip','not recorded'))
else:
    print('Task 2.6 metrics JSON not found at:', TASK26_METRICS_JSON)
if TASK26_MANIFEST.is_file():
    with TASK26_MANIFEST.open(newline='',encoding='utf-8-sig') as f: manifest_rows=list(csv.DictReader(f))
    print('Dataset image counts by source/split:', dict(Counter((r.get('source'),r.get('split')) for r in manifest_rows)))
    print('Dataset image counts by split:', dict(Counter(r.get('split') for r in manifest_rows)))
else:
    print('Task 2.6 manifest not found at:', TASK26_MANIFEST)


## ByteTrack runs — four JSONL outputs

Each model/video pair gets its own fresh tracker. The COCO baseline only emits classes with a defensible mapping: bicycle/motorcycle → `two_wheeler`, car → `car`, bus/truck → `bus_truck`, person → `pedestrian`. It cannot identify `auto`, `e_rickshaw`, or `cart`; those categories are available only from the trained VEYAKSH checkpoint if it learned them. A detection without an assigned ID is retained with `track_id: null`.


In [ ]:
TARGET_NAMES = {0:'two_wheeler',1:'auto',2:'e_rickshaw',3:'car',4:'bus_truck',5:'cart',6:'pedestrian'}
COCO_TO_TARGET = {'bicycle':'two_wheeler','motorcycle':'two_wheeler','car':'car','bus':'bus_truck','truck':'bus_truck','person':'pedestrian'}

def as_name_map(names):
    return {int(k):str(v) for k,v in names.items()} if isinstance(names,dict) else {i:str(v) for i,v in enumerate(names)}

def export_tracks(video_path, weights, kind, output_path):
    model=YOLO(str(weights)); name_map=as_name_map(model.names)
    if kind=='coco':
        class_map={i:COCO_TO_TARGET[n] for i,n in name_map.items() if n in COCO_TO_TARGET}
    else:
        aliases={'bike':'two_wheeler','motorcycle':'two_wheeler','motorbike':'two_wheeler','rickshaw':'auto','auto_rickshaw':'auto','erickshaw':'e_rickshaw','e_rickshaw':'e_rickshaw','bus':'bus_truck','truck':'bus_truck','tempo':'bus_truck','person':'pedestrian'}
        class_map={i:(n if n in TARGET_NAMES.values() else aliases.get(n)) for i,n in name_map.items()}
        class_map={i:n for i,n in class_map.items() if n in TARGET_NAMES.values()}
        assert class_map, f'No target classes in {weights}: {name_map}'
    cap=cv2.VideoCapture(str(video_path)); fps=float(cap.get(cv2.CAP_PROP_FPS)); cap.release()
    assert fps>0, f'Invalid video FPS: {video_path}'
    processed=0; written=0
    with Path(output_path).open('w',encoding='utf-8') as out:
        stream=model.track(source=str(video_path),tracker='bytetrack.yaml',persist=True,stream=True,verbose=False,device=DEVICE,imgsz=640,conf=0.15,classes=sorted(class_map))
        for result in stream:
            frame=processed; processed+=1
            b=result.boxes
            if b is None or len(b)==0: continue
            coords=b.xyxy.cpu().tolist(); cids=b.cls.cpu().tolist(); confs=b.conf.cpu().tolist()
            tids=b.id.cpu().tolist() if b.id is not None else [None]*len(coords)
            for xy,cid,conf,tid in zip(coords,cids,confs,tids):
                target=class_map.get(int(cid))
                if target is None: continue
                out.write(json.dumps({'frame':frame,'ts':round(frame/fps,6),'track_id':int(tid) if tid is not None else None,'class':target,'bbox':[round(float(x),2) for x in xy],'conf':round(float(conf),6)},separators=(',',':'))+'\n')
                written+=1
            if processed%500==0: print(f'{video_path.name} / {weights}: {processed} frames, {written} detections')
    print(f'Saved {written} rows from {processed} frames: {output_path}')

import urllib.request
COCO_WEIGHTS = VEYAKSH/'models/yolo11n.pt'
if not COCO_WEIGHTS.exists():
    urllib.request.urlretrieve('https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt',COCO_WEIGHTS)
for video_name, video in [('test',TEST_VIDEO),('campus',CAMPUS_VIDEO)]:
    export_tracks(video,COCO_WEIGHTS,'coco',TRACKS_DIR/f'{video_name}_yolo11n_tracks.jsonl')
    export_tracks(video,BEST,'veyaksh',TRACKS_DIR/f'{video_name}_best_tracks.jsonl')
print('Four JSONL files:', *[p.name for p in sorted(TRACKS_DIR.glob('*.jsonl'))], sep='\n - ')


## Completion check

Drive outputs expected after successful run:
- `VEYAKSH_TRAFFIC_DATASET/models/task2_7_metrics.csv`
- `VEYAKSH_TRAFFIC_DATASET/models/task2_7_eval/heldout_clip_eval/confusion_matrix.png` (and normalized plot)
- `VEYAKSH_TRAFFIC_DATASET/tracks/test_yolo11n_tracks.jsonl`
- `VEYAKSH_TRAFFIC_DATASET/tracks/test_best_tracks.jsonl`
- `VEYAKSH_TRAFFIC_DATASET/tracks/campus_yolo11n_tracks.jsonl`
- `VEYAKSH_TRAFFIC_DATASET/tracks/campus_best_tracks.jsonl`

Update `m2_detect_track/DECISIONS.md` with the exact validation clip ID, split counts, class-wise AP, and limitations. The final task also requires a Git push/MR to the correct `anant/...` repository; this checkout currently points to a different GitLab remote, so confirm the intended target before publishing.
